In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


# Inspect the directory structure inside /kaggle/input
print("Mounted datasets under /kaggle/input:")
for item in os.listdir('/kaggle/input'):
    print(" -", item)

# Search for the directory containing .h5 slice files
target_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'ACDC_training_slices' in dirs:
        target_dir = os.path.join(root, 'ACDC_training_slices')
        break
    elif any(f.endswith('.h5') for f in files):
        target_dir = root
        break

print(f"\nResolved training slices directory: {target_dir}")

In [ ]:
import os
import h5py
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

def find_slices_dir(search_root="/kaggle/input"):
    """Recursively search for the ACDC_training_slices directory."""
    for root, dirs, files in os.walk(search_root):
        if 'ACDC_training_slices' in dirs:
            return os.path.join(root, 'ACDC_training_slices')
    for root, dirs, files in os.walk(search_root):
        if any(f.endswith('.h5') for f in files):
            return root
    raise FileNotFoundError(f"Could not find any directory containing .h5 files under {search_root}")

class ACDC2DDataset(Dataset):
    def __init__(self, target_size=(256, 256)):
        self.slice_dir = find_slices_dir()
        self.target_size = target_size
        self.sample_list = sorted([f for f in os.listdir(self.slice_dir) if f.endswith('.h5')])
        print(f"Successfully loaded {len(self.sample_list)} slices from: {self.slice_dir}")
        
    def __len__(self):
        return len(self.sample_list)
        
    def __getitem__(self, idx):
        file_name = self.sample_list[idx]
        file_path = os.path.join(self.slice_dir, file_name)
        
        with h5py.File(file_path, 'r') as h5f:
            image = h5f['image'][:]
            label = h5f['label'][:]
        
        # 1. Convert to PyTorch tensors and add dummy batch/channel dimensions (1, 1, H, W)
        # Interpolate function requires a 4D tensor.
        img_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        lbl_tensor = torch.tensor(label, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        
        # 2. Apply Resizing to enforce uniform dimensions
        # Use Bilinear for the MRI Image to keep it smooth
        img_resized = F.interpolate(img_tensor, size=self.target_size, mode='bilinear', align_corners=False)
        
        # Use Nearest Neighbor for the Mask to preserve exact integer classes (0, 1, 2, 3)
        lbl_resized = F.interpolate(lbl_tensor, size=self.target_size, mode='nearest')
        
        # 3. Strip the dummy batch dimension
        # Final image shape: [1, 256, 256] | Final label shape: [256, 256]
        final_image = img_resized.squeeze(0)
        final_label = lbl_resized.squeeze(0).squeeze(0).long()
        
        return final_image, final_label

# Initialize Dataset and DataLoader
train_dataset = ACDC2DDataset(target_size=(256, 256))
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Verify the first batch
images, labels = next(iter(train_loader))
print(f"Success! Batch images shape: {images.shape}")
print(f"Success! Batch labels shape: {labels.shape}")

# Plot to verify structural integrity is maintained after resize
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Resized MRI Slice (256x256)")
plt.imshow(images[0].squeeze(), cmap='gray')

plt.subplot(1, 2, 2)
plt.title("Resized Ground Truth Mask (256x256)")
plt.imshow(labels[0], cmap='jet')
plt.show()

In [ ]:
# Inspect the keys and attributes of a single .h5 file
sample_path = os.path.join(train_dataset.slice_dir, train_dataset.sample_list[0])
with h5py.File(sample_path, 'r') as h5f:
    print(f"File: {train_dataset.sample_list[0]}")
    print("Keys inside the HDF5 file:", list(h5f.keys()))
    print("Attributes (metadata):", dict(h5f.attrs))

### Shared 2D Encoder-Decoder Architecture.

In [ ]:
import torch.nn as nn

class SharedEncoderMultiTaskNet(nn.Module):
    def __init__(self):
        super(SharedEncoderMultiTaskNet, self).__init__()
        
        # 1. Shared 2D Encoder (Feature Extractor)
        # Compresses 256x256 -> 32x32
        self.encoder = nn.Sequential(
            self._conv_block(1, 32),
            nn.MaxPool2d(2), # 128x128
            self._conv_block(32, 64),
            nn.MaxPool2d(2), # 64x64
            self._conv_block(64, 128),
            nn.MaxPool2d(2), # 32x32
            self._conv_block(128, 256)
        )
        
        # 2. Branch A: Segmentation Decoder (The Spatial Mask)
        # Expands 32x32 back to 256x256
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
            self._conv_block(128, 128),
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            self._conv_block(64, 64),
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            self._conv_block(32, 32),
            # Output: 4 Channels (Background, LV, RV, MYO)
            nn.Conv2d(32, 4, kernel_size=1) 
        )
        
        # 3. Branch B: Clinical Classifier (The Disease Diagnosis)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            # Output: 5 Classes (Normal, MINF, DCM, HCM, RV)
            nn.Linear(64, 5) 
        )

    def _conv_block(self, in_c, out_c):
        """Helper function for standard Conv -> BatchNorm -> ReLU"""
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # Pass input through the shared backbone
        latent_features = self.encoder(x)
        
        # Branch A: Generate the anatomical masks
        segmentation_mask = self.decoder(latent_features)
        
        # Branch B: Generate the clinical classification
        pooled = self.global_pool(latent_features).view(x.size(0), -1)
        clinical_class = self.classifier(pooled)
        
        return segmentation_mask, clinical_class

# Initialize the model and pass your successfully loaded batch through it
model = SharedEncoderMultiTaskNet()
mask_preds, class_preds = model(images)

print(f"Input Images Shape: {images.shape}")
print(f"Output Mask Predictions Shape: {mask_preds.shape} -> [Batch, 4 Classes, H, W]")
print(f"Output Clinical Predictions Shape: {class_preds.shape} -> [Batch, 5 Disease Classes]")

### We are now ready to tackle Objective 2: The Directional Gradient Boundary Penalty (L topo).
This is the core scientific novelty of your thesis. Instead of simply comparing whether Pixel A matches Pixel A, this loss function calculates the spatial derivative (the geometric edge transitions) of the heart in both the X and Y directions. If the network tries to draw a broken myocardial ring, the gradient vectors of the prediction will violently misalign with the continuous gradient vectors of the ground truth, resulting in a massive penalty.
The PyTorch Implementation of L topo	
 
We will use highly efficient Sobel operators as 2D convolutional kernels. Because these kernels are fixed mathematical filters, they do not require parameter training, keeping your model extremely lightweight.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DirectionalGradientLoss(nn.Module):
    def __init__(self):
        super(DirectionalGradientLoss, self).__init__()
        
        # 1. Define fixed Sobel kernels for X and Y spatial gradients
        sobel_x = torch.tensor([[-1, 0, 1], 
                                [-2, 0, 2], 
                                [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
                                
        sobel_y = torch.tensor([[-1, -2, -1], 
                                [ 0,  0,  0], 
                                [ 1,  2,  1]], dtype=torch.float32).view(1, 1, 3, 3)
        
        # Register as buffers so they automatically move to the GPU (cuda) with the model
        # and are NOT updated by the optimizer.
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def forward(self, preds, targets):
        """
        preds: [Batch, 4, H, W] - Raw logits from the segmentation decoder
        targets: [Batch, H, W]  - Ground truth integer masks (0, 1, 2, 3)
        """
        # Convert raw logits to probabilities (0.0 to 1.0)
        preds_prob = F.softmax(preds, dim=1)
        
        # Convert integer masks into 4-channel One-Hot encodings to match predictions
        # Shape becomes [Batch, 4, H, W]
        targets_one_hot = F.one_hot(targets, num_classes=4).permute(0, 3, 1, 2).float()
        
        loss = 0.0
        
        # Calculate topological edge gradients for each anatomical class independently
        for c in range(4):
            # Isolate the channel for class 'c' -> Shape: [Batch, 1, H, W]
            pred_c = preds_prob[:, c:c+1, :, :]
            target_c = targets_one_hot[:, c:c+1, :, :]
            
            # Apply Sobel filters to extract spatial boundaries
            pred_grad_x = F.conv2d(pred_c, self.sobel_x, padding=1)
            pred_grad_y = F.conv2d(pred_c, self.sobel_y, padding=1)
            
            target_grad_x = F.conv2d(target_c, self.sobel_x, padding=1)
            target_grad_y = F.conv2d(target_c, self.sobel_y, padding=1)
            
            # Calculate the Mean Squared Error between the predicted edges and true edges
            loss += F.mse_loss(pred_grad_x, target_grad_x) + F.mse_loss(pred_grad_y, target_grad_y)
            
        # Average the loss across the 4 classes
        return loss / 4.0

# --- Test the Loss Function Immediately ---
# We will use the 'mask_preds' and 'labels' you generated in the previous step
topo_criterion = DirectionalGradientLoss()

# Calculate the topological error of your randomly initialized network
topology_loss_value = topo_criterion(mask_preds, labels)

print(f"Calculated L_topo penalty: {topology_loss_value.item():.4f}")

### The Joint-Optimization Strategy (L total)
When you assemble your final training loop, you will combine this topological loss with standard pixel-wise segmentation loss and categorical disease loss.
Your total loss equation L total=αLseg+βLclass+γLtopo	
  will look exactly like this in your training step:

In [ ]:
# Standard Loss Functions
seg_criterion = nn.CrossEntropyLoss()
class_criterion = nn.CrossEntropyLoss()

# --- THE FIX: Generate dummy clinical labels for the 16-image batch ---
clinical_labels = torch.randint(0, 5, (16,)) 

# Alpha, Beta, Gamma hyperparameters
alpha = 1.0
beta = 0.5
gamma = 0.1 

# The Combined Mathematical Penalty
L_seg = seg_criterion(mask_preds, labels)
L_class = class_criterion(class_preds, clinical_labels) # This will now work
L_topo = topo_criterion(mask_preds, labels)

L_total = (alpha * L_seg) + (beta * L_class) + (gamma * L_topo)

print(f"L_seg: {L_seg.item():.4f}")
print(f"L_class: {L_class.item():.4f}")
print(f"L_topo: {L_topo.item():.4f}")
print(f"L_total: {L_total.item():.4f}")

# Backpropagation step
L_total.backward()
print("Success! Gradients calculated and backpropagated.")

In [ ]:
import os
import h5py

sample_path = os.path.join(train_dataset.slice_dir, train_dataset.sample_list[0])
with h5py.File(sample_path, 'r') as h5f:
    print(f"File Name: {train_dataset.sample_list[0]}")
    print("Data Keys inside HDF5:", list(h5f.keys()))
    print("Metadata Attributes:", dict(h5f.attrs))

In [ ]:
import os

dataset_root = "/kaggle/input/"
metadata_files = []

# Scan the Kaggle directory for any patient mapping files
for root, dirs, files in os.walk(dataset_root):
    for file in files:
        if file.endswith(('.csv', '.txt', '.cfg', '.xlsx', '.json')):
            metadata_files.append(os.path.join(root, file))

print("Found potential clinical mapping files:")
for f in metadata_files:
    print(" -", f)
import os
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

# ---------------------------------------------------------
# 1. THE COMPLETE DATASET PIPELINE
# ---------------------------------------------------------
class ACDC2DDataset(Dataset):
    def __init__(self, target_size=(256, 256)):
        # Locate Kaggle directory
        self.slice_dir = None
        for root, dirs, files in os.walk("/kaggle/input"):
            if 'ACDC_training_slices' in dirs:
                self.slice_dir = os.path.join(root, 'ACDC_training_slices')
                break
        if not self.slice_dir:
            for root, dirs, files in os.walk("/kaggle/input"):
                if any(f.endswith('.h5') for f in files):
                    self.slice_dir = root
                    break
                    
        self.target_size = target_size
        self.sample_list = sorted([f for f in os.listdir(self.slice_dir) if f.endswith('.h5')])
        
    def _get_clinical_class(self, patient_id_str):
        """Mathematically maps patient ID (e.g., 'patient045') to classes 0-4"""
        patient_num = int(patient_id_str.replace("patient", ""))
        if patient_num <= 20: return 0   # Normal
        if patient_num <= 40: return 1   # MINF
        if patient_num <= 60: return 2   # DCM
        if patient_num <= 80: return 3   # HCM
        return 4                         # ARV

    def __len__(self):
        return len(self.sample_list)
        
    def __getitem__(self, idx):
        file_name = self.sample_list[idx]
        file_path = os.path.join(self.slice_dir, file_name)
        
        # Extract Patient ID (e.g., 'patient001_frame01_slice_0.h5' -> 'patient001')
        patient_id = file_name.split('_')[0]
        clinical_label = self._get_clinical_class(patient_id)
        
        with h5py.File(file_path, 'r') as h5f:
            image = h5f['image'][:]
            label = h5f['label'][:]
        
        # Convert and Resize Tensors
        img_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        lbl_tensor = torch.tensor(label, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        
        img_resized = F.interpolate(img_tensor, size=self.target_size, mode='bilinear', align_corners=False)
        lbl_resized = F.interpolate(lbl_tensor, size=self.target_size, mode='nearest')
        
        final_image = img_resized.squeeze(0)
        final_label = lbl_resized.squeeze(0).squeeze(0).long()
        final_clinical = torch.tensor(clinical_label, dtype=torch.long)
        
        # Now yields: (Image, Segmentation Mask, Disease Class)
        return final_image, final_label, final_clinical

# Initialize DataLoader
# Increase batch_size to 32 (16 images per GPU) and utilize multi-core CPU loading
train_loader = DataLoader(
    ACDC2DDataset(target_size=(256, 256)), 
    batch_size=32,       # Doubled for dual GPUs
    shuffle=True, 
    num_workers=2,       # Uses Kaggle's dual CPU cores to load images faster
    pin_memory=True      # Speeds up CPU to GPU memory transfers
)
# ---------------------------------------------------------
# 2. DEVICE SETUP & MODEL INITIALIZATION
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on Device: {device}")

# (Assuming SharedEncoderMultiTaskNet and DirectionalGradientLoss are defined from previous steps)
model = SharedEncoderMultiTaskNet().to(device)

seg_criterion = nn.CrossEntropyLoss()
class_criterion = nn.CrossEntropyLoss()
topo_criterion = DirectionalGradientLoss().to(device)

optimizer = Adam(model.parameters(), lr=1e-4)

# ---------------------------------------------------------
# 3. THE MULTI-TASK TRAINING LOOP (1 Epoch Demo)
# ---------------------------------------------------------
print("\n--- Starting Phase 1 Training Loop ---")
model.train()

alpha, beta, gamma = 1.0, 0.5, 0.1  # Loss balancing weights
total_batches = len(train_loader)

for batch_idx, (images, masks, clinical_classes) in enumerate(train_loader):
    # Move data to GPU
    images = images.to(device)
    masks = masks.to(device)
    clinical_classes = clinical_classes.to(device)
    
    # Forward Pass
    optimizer.zero_grad()
    mask_preds, class_preds = model(images)
    
    # Calculate Individual Losses
    L_seg = seg_criterion(mask_preds, masks)
    L_class = class_criterion(class_preds, clinical_classes)
    L_topo = topo_criterion(mask_preds, masks)
    
    # Joint Optimization Target
    L_total = (alpha * L_seg) + (beta * L_class) + (gamma * L_topo)
    
    # Backpropagation
    L_total.backward()
    optimizer.step()
    
    # Print progress every 20 batches
    if batch_idx % 20 == 0:
        print(f"Batch {batch_idx:03d}/{total_batches} | "
              f"L_seg: {L_seg.item():.3f} | L_class: {L_class.item():.3f} | L_topo: {L_topo.item():.3f} | L_total: {L_total.item():.3f}")

print("\nSuccess! Multi-task joint optimization loop is fully functional.")

In [ ]:
## For model performance evaluation
import torch

def calculate_dice(preds, targets, num_classes=4):
    """Calculates the Dice Similarity Coefficient for each anatomical class."""
    # Convert raw logits to class predictions (0, 1, 2, 3)
    preds = torch.argmax(preds, dim=1) 
    dice_scores = []
    
    for c in range(1, num_classes): # Skip background (class 0)
        pred_c = (preds == c)
        target_c = (targets == c)
        
        intersection = (pred_c & target_c).float().sum()
        union = pred_c.float().sum() + target_c.float().sum()
        
        # Add epsilon to prevent division by zero
        dice = (2. * intersection + 1e-7) / (union + 1e-7)
        dice_scores.append(dice.item())
        
    # Return average Dice across LV, RV, and MYO
    return sum(dice_scores) / len(dice_scores)

def calculate_accuracy(preds, targets):
    """Calculates disease classification accuracy."""
    preds_classes = torch.argmax(preds, dim=1)
    correct = (preds_classes == targets).float().sum()
    return (correct / len(targets)).item()

We will wrap training loop into a reusable Python function. This allows us to call train_ablation_model(gamma_weight=0.0) for your baseline, and then train_ablation_model(gamma_weight=0.1) for your novel framework.

In [ ]:
def train_ablation_model(gamma_weight, num_epochs=20):
    print(f"\n========== STARTING RUN: Gamma (L_topo) = {gamma_weight} ==========")
    
    # Re-initialize a fresh model and optimizer for a fair test
    test_model = SharedEncoderMultiTaskNet().to(device)
    test_optimizer = Adam(test_model.parameters(), lr=1e-4)
    
    # History trackers
    history = {'loss': [], 'dice': [], 'accuracy': []}
    
    for epoch in range(num_epochs):
        test_model.train()
        epoch_seg_loss = 0
        epoch_class_loss = 0
        epoch_topo_loss = 0
        
        epoch_dice = 0
        epoch_acc = 0
        
        for images, masks, clinical_classes in train_loader:
            images, masks, clinical_classes = images.to(device), masks.to(device), clinical_classes.to(device)
            
            test_optimizer.zero_grad()
            mask_preds, class_preds = test_model(images)
            
            L_seg = seg_criterion(mask_preds, masks)
            L_class = class_criterion(class_preds, clinical_classes)
            L_topo = topo_criterion(mask_preds, masks)
            
            # THE ABLATION SWITCH
            L_total = (1.0 * L_seg) + (0.5 * L_class) + (gamma_weight * L_topo)
            
            L_total.backward()
            test_optimizer.step()
            
            # Track metrics
            epoch_seg_loss += L_seg.item()
            epoch_class_loss += L_class.item()
            epoch_topo_loss += L_topo.item()
            epoch_dice += calculate_dice(mask_preds, masks)
            epoch_acc += calculate_accuracy(class_preds, clinical_classes)
            
        # Average metrics over the epoch
        batches = len(train_loader)
        avg_dice = epoch_dice / batches
        avg_acc = epoch_acc / batches
        
        history['dice'].append(avg_dice)
        history['accuracy'].append(avg_acc)
        
        print(f"Epoch {epoch+1:02d}/{num_epochs} | "
              f"Dice: {avg_dice:.4f} | Acc: {avg_acc*100:.1f}% | "
              f"L_seg: {epoch_seg_loss/batches:.3f} | L_class: {epoch_class_loss/batches:.3f} | L_topo: {epoch_topo_loss/batches:.3f}")
              
    return history, test_model

In [ ]:
#Now, you simply execute both runs and capture their histories. Run this cell to start the formal comparison. 
# (Note: We are running 20 epochs for the initial test. You can scale this to 50 or 100 later).
# 1. Train the Baseline Model (Standard Naive Multi-tasking)
print("Training Model A: No Topological Constraint")
history_baseline = train_ablation_model(gamma_weight=0.0, num_epochs=20)

# 2. Train Your Proposed Model (Topology-Preserved)
print("\nTraining Model B: Topology-Preserved Framework")
history_proposed = train_ablation_model(gamma_weight=0.1, num_epochs=20)

For Multi-GPU setup Changed the Mask architecture from SharedEncoderMultiTaskNet to multiTaskUnet 

In [ ]:
import torch
import numpy as np
import random
import os

def seed_everything(seed=42):
    """Locks all random number generators for absolute reproducibility."""
    # 1. Set Python, Numpy, and OS seeds
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. Set PyTorch seeds
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # For Multi-GPU
    
    # 3. Force CuDNN to behave deterministically (slightly slower, but reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"Global seed locked to {seed}. Training is now 100% deterministic.")

seed_everything(42)

In [ ]:
# import copy
# import torch
# import torch.nn.functional as F
# import numpy as np
# import h5py
# import os
# from torch.utils.data import Dataset, DataLoader

# class ACDC2DDataset(Dataset):
#     def __init__(self, target_size=(256, 256)):
#         self.slice_dir = None
#         for root, dirs, files in os.walk("/kaggle/input"):
#             if 'ACDC_training_slices' in dirs:
#                 self.slice_dir = os.path.join(root, 'ACDC_training_slices')
#                 break
#         if not self.slice_dir:
#             for root, dirs, files in os.walk("/kaggle/input"):
#                 if any(f.endswith('.h5') for f in files):
#                     self.slice_dir = root
#                     break
                    
#         self.target_size = target_size
#         self.sample_list = sorted([f for f in os.listdir(self.slice_dir) if f.endswith('.h5')])
        
#     def _get_clinical_class(self, patient_id_str):
#         patient_num = int(patient_id_str.replace("patient", ""))
#         if patient_num <= 20: return 0   
#         if patient_num <= 40: return 1   
#         if patient_num <= 60: return 2   
#         if patient_num <= 80: return 3   
#         return 4                         

#     def __len__(self):
#         return len(self.sample_list)
        
#     def __getitem__(self, idx):
#         file_name = self.sample_list[idx]
#         file_path = os.path.join(self.slice_dir, file_name)
        
#         patient_id = file_name.split('_')[0]
#         clinical_label = self._get_clinical_class(patient_id)
        
#         with h5py.File(file_path, 'r') as h5f:
#             image = h5f['image'][:]
#             label = h5f['label'][:]
            
#         # THE FIX: Z-Score Normalization for MRI Intensities
#         image = (image - np.mean(image)) / (np.std(image) + 1e-8)
        
#         img_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
#         lbl_tensor = torch.tensor(label, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        
#         img_resized = F.interpolate(img_tensor, size=self.target_size, mode='bilinear', align_corners=False)
#         lbl_resized = F.interpolate(lbl_tensor, size=self.target_size, mode='nearest')
        
#         return img_resized.squeeze(0), lbl_resized.squeeze(0).squeeze(0).long(), torch.tensor(clinical_label, dtype=torch.long)

# # 1. Create the base dataset
# full_dataset = ACDC2DDataset()

# # 2. Perform Strict 80/20 Patient Split
# train_samples, val_samples = [], []
# for f in full_dataset.sample_list:
#     p_num = int(f.split('_')[0].replace("patient", ""))
#     # Last 4 patients of each 20-patient class go to validation
#     if p_num % 20 in [17, 18, 19, 0]: 
#         val_samples.append(f)
#     else:
#         train_samples.append(f)

# # 3. Initialize loaders
# train_dataset = copy.deepcopy(full_dataset)
# val_dataset = copy.deepcopy(full_dataset)
# train_dataset.sample_list = train_samples
# val_dataset.sample_list = val_samples

# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
# val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# print(f"Data ready. Training on {len(train_samples)} slices, Validating on {len(val_samples)} UNSEEN slices.")

In [ ]:
import copy
from torch.utils.data import DataLoader

# Assuming your original full dataset is initialized
full_dataset = ACDC2DDataset(target_size=(256, 256))

train_samples = []
val_samples = []

# Perform a strict patient-level split (No cross-contamination)
for f in full_dataset.sample_list:
    # Extract patient number (e.g., 'patient017' -> 17)
    p_num = int(f.split('_')[0].replace("patient", ""))
    
    # Validation condition: The last 4 patients of every 20-patient class bracket
    # Normal (17-20), MINF (37-40), DCM (57-60), HCM (77-80), ARV (97-100)
    if p_num % 20 in [17, 18, 19, 0]: 
        val_samples.append(f)
    else:
        train_samples.append(f)

# Clone the dataset object to create independent train and val sets
train_dataset = copy.deepcopy(full_dataset)
val_dataset = copy.deepcopy(full_dataset)

train_dataset.sample_list = train_samples
val_dataset.sample_list = val_samples

print(f"Strict Patient-Level Split Complete:")
print(f" - Training Slices: {len(train_dataset.sample_list)} (80 Patients)")
print(f" - Validation Slices: {len(val_dataset.sample_list)} (20 Patients)")

# Initialize the new DataLoaders
# Validation does not need shuffling
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import torch
import torch.nn as nn

class MultiTaskUNet(nn.Module):
    def __init__(self):
        super(MultiTaskUNet, self).__init__()
        
        # ---------------- ENCODER ----------------
        self.enc1 = self._conv_block(1, 32)
        self.pool1 = nn.MaxPool2d(2) # 128x128
        
        self.enc2 = self._conv_block(32, 64)
        self.pool2 = nn.MaxPool2d(2) # 64x64
        
        self.enc3 = self._conv_block(64, 128)
        self.pool3 = nn.MaxPool2d(2) # 32x32
        
        # BOTTLENECK (Latent Space)
        self.bottleneck = self._conv_block(128, 256)
        
        # ---------------- DECODER (Branch A: Segmentation) ----------------
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        # Note the input channels are doubled (128 from upsample + 128 from skip connection)
        self.dec3 = self._conv_block(256, 128) 
        
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(128, 64)
        
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(64, 32)
        
        # Final Mask Output (4 Channels)
        self.mask_out = nn.Conv2d(32, 4, kernel_size=1)
        
        # ---------------- CLASSIFIER (Branch B: Clinical Disease) ----------------
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, 5) # 5 Cardiomyopathy Classes
        )

    def _conv_block(self, in_c, out_c):
        """Standard U-Net Convolutional Block (Double Conv)"""
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # 1. Encoder (Save features for skip connections)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        
        b = self.bottleneck(self.pool3(e3))
        
        # 2. Decoder (Concatenate the skip connections)
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1) # The Skip Connection
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1) # The Skip Connection
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1) # The Skip Connection
        d1 = self.dec1(d1)
        
        mask_preds = self.mask_out(d1)
        
        # 3. Classifier (Fed by the bottleneck)
        pooled = self.global_pool(b).view(x.size(0), -1)
        class_preds = self.classifier(pooled)
        
        return mask_preds, class_preds

In [ ]:
# import torch.nn as nn
# from torch.optim import Adam

# def train_and_validate_model(gamma_weight, model_name, num_epochs=30):
#     print(f"\n========== STARTING FULL RUN: {model_name} (Gamma = {gamma_weight}) ==========")
    
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
#     # ALWAYS initialize a fresh model to prevent memory state issues
#     model = MultiTaskUNet()
#     if torch.cuda.device_count() > 1:
#         model = nn.DataParallel(model)
#     model = model.to(device)
    
#     optimizer = Adam(model.parameters(), lr=1e-4)
    
#     for epoch in range(num_epochs):
#         # --- TRAINING PHASE ---
#         model.train()
#         epoch_dice, epoch_acc = 0, 0
        
#         for images, masks, clinical_classes in train_loader:
#             images, masks, clinical_classes = images.to(device), masks.to(device), clinical_classes.to(device)
#             optimizer.zero_grad()
            
#             mask_preds, class_preds = model(images)
            
#             L_seg = seg_criterion(mask_preds, masks)
#             L_class = class_criterion(class_preds, clinical_classes)
#             L_topo = topo_criterion(mask_preds, masks)
            
#             L_total = (1.0 * L_seg) + (0.5 * L_class) + (gamma_weight * L_topo)
            
#             L_total.backward()
#             optimizer.step()
            
#             epoch_dice += calculate_dice(mask_preds, masks)
#             epoch_acc += calculate_accuracy(class_preds, clinical_classes)
            
#         train_dice = epoch_dice / len(train_loader)
#         train_acc = epoch_acc / len(train_loader)
        
#         # --- VALIDATION PHASE (On Unseen Patients) ---
#         model.eval()
#         val_dice, val_acc = 0, 0
#         with torch.no_grad():
#             for images, masks, clinical_classes in val_loader:
#                 images, masks, clinical_classes = images.to(device), masks.to(device), clinical_classes.to(device)
#                 mask_preds, class_preds = model(images)
                
#                 val_dice += calculate_dice(mask_preds, masks)
#                 val_acc += calculate_accuracy(class_preds, clinical_classes)
                
#         val_dice = val_dice / len(val_loader)
#         val_acc = val_acc / len(val_loader)
        
#         print(f"Epoch {epoch+1:02d}/{num_epochs} | "
#               f"TRAIN [Dice: {train_dice:.3f}, Acc: {train_acc*100:.1f}%] | "
#               f"VAL [Dice: {val_dice:.3f}, Acc: {val_acc*100:.1f}%]")
              
#     return model

# # 1. Train Model A (Baseline)
# model_A_final = train_and_validate_model(gamma_weight=0.0, model_name="Model A (Baseline)", num_epochs=30)

# # 2. Train Model B (Topology)
# model_B_final = train_and_validate_model(gamma_weight=0.1, model_name="Model B (Topology)", num_epochs=30)

In [ ]:

import torch
import torch.nn as nn
from torch.optim import Adam

def train_ablation_model_multigpu(gamma_weight, num_epochs=20):
    print(f"\n========== STARTING MULTI-GPU RUN: Gamma (L_topo) = {gamma_weight} ==========")
    
    # 1. Device and Multi-GPU Setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Initialize the base model
    test_model = MultiTaskUNet()
    
    # Check for multiple GPUs and wrap the model
    if torch.cuda.device_count() > 1:
        print(f"Hardware detected: Utilizing {torch.cuda.device_count()} GPUs via DataParallel!")
        test_model = nn.DataParallel(test_model)
        
    test_model = test_model.to(device)
    test_optimizer = Adam(test_model.parameters(), lr=1e-4)
    
    # History trackers
    history = {'loss': [], 'dice': [], 'accuracy': []}
    
    for epoch in range(num_epochs):
        test_model.train()
        epoch_seg_loss, epoch_class_loss, epoch_topo_loss = 0, 0, 0
        epoch_dice, epoch_acc = 0, 0
        
        for images, masks, clinical_classes in train_loader:
            # Send data to the primary device (DataParallel will handle distribution)
            images = images.to(device)
            masks = masks.to(device)
            clinical_classes = clinical_classes.to(device)
            
            test_optimizer.zero_grad()
            
            # Forward pass is automatically split across both GPUs
            mask_preds, class_preds = test_model(images)
            
            # Calculate Losses
            L_seg = seg_criterion(mask_preds, masks)
            L_class = class_criterion(class_preds, clinical_classes)
            L_topo = topo_criterion(mask_preds, masks)
            
            # Joint Optimization Target
            L_total = (1.0 * L_seg) + (0.5 * L_class) + (gamma_weight * L_topo)
            
            # Synchronized Backpropagation
            L_total.backward()
            test_optimizer.step()
            
            # Track metrics
            epoch_seg_loss += L_seg.item()
            epoch_class_loss += L_class.item()
            epoch_topo_loss += L_topo.item()
            epoch_dice += calculate_dice(mask_preds, masks)
            epoch_acc += calculate_accuracy(class_preds, clinical_classes)
            
        # Average metrics over the epoch
        batches = len(train_loader)
        avg_dice = epoch_dice / batches
        avg_acc = epoch_acc / batches
        
        history['dice'].append(avg_dice)
        history['accuracy'].append(avg_acc)
        
        print(f"Epoch {epoch+1:02d}/{num_epochs} | "
              f"Dice: {avg_dice:.4f} | Acc: {avg_acc*100:.1f}% | "
              f"L_seg: {epoch_seg_loss/batches:.3f} | L_class: {epoch_class_loss/batches:.3f} | L_topo: {epoch_topo_loss/batches:.3f}")
              
    return history, test_model
# Capture BOTH the history and the trained model
print("Training Model A: Standard Naive Multi-tasking")
history_baseline, model_A = train_ablation_model_multigpu(gamma_weight=0.0, num_epochs=30)

print("\nTraining Model B: Topology-Preserved Framework")
history_proposed, model_B = train_ablation_model_multigpu(gamma_weight=0.1, num_epochs=30)

In [ ]:
import torch

def save_model(model, filepath):
    # Check if the model is wrapped in DataParallel
    if isinstance(model, torch.nn.DataParallel):
        # Extract the raw model state dictionary
        state_dict = model.module.state_dict()
    else:
        state_dict = model.state_dict()
        
    torch.save(state_dict, filepath)
    print(f"Successfully saved weights to: {filepath}")

# Save both models
save_model(model_A, '/kaggle/working/Model_A_Baseline_Weights.pth')
save_model(model_B, '/kaggle/working/Model_B_Topology_Weights.pth')

print("Models are saved. You can download them from the Kaggle 'Output' sidebar.")

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Epochs (1 to 30)
# epochs = np.arange(1, 31)

# # Extracted Data: Model A (Baseline)
# dice_A = [0.0121, 0.0008, 0.0002, 0.0039, 0.0384, 0.1049, 0.2229, 0.2774, 0.2892, 0.3078, 0.3156, 0.3378, 0.3768, 0.4257, 0.4419, 0.4640, 0.4810, 0.5194, 0.5378, 0.5776, 0.6196, 0.6679, 0.6971, 0.7274, 0.7392, 0.7434, 0.7506, 0.7580, 0.7633, 0.7713]
# acc_A = [22.3, 34.3, 35.9, 38.4, 41.1, 45.1, 47.2, 47.7, 49.4, 52.4, 53.4, 57.1, 59.2, 58.5, 60.0, 61.9, 63.7, 66.4, 65.7, 68.1, 68.9, 70.2, 71.2, 73.5, 73.5, 76.2, 78.3, 76.7, 77.8, 81.2]

# # Extracted Data: Model B (Topology-Preserved)
# dice_B = [0.0465, 0.0803, 0.0992, 0.1221, 0.1610, 0.2049, 0.2201, 0.2439, 0.2546, 0.2688, 0.2818, 0.3076, 0.3308, 0.3598, 0.4115, 0.4949, 0.5828, 0.6353, 0.6705, 0.6951, 0.7117, 0.7146, 0.7342, 0.7379, 0.7509, 0.7539, 0.7518, 0.7623, 0.7655, 0.7684]
# acc_B = [24.6, 31.3, 34.1, 37.6, 41.2, 44.8, 48.0, 49.4, 48.5, 52.3, 54.3, 56.2, 55.8, 59.3, 58.4, 61.8, 63.6, 64.0, 65.7, 69.6, 69.4, 70.0, 73.1, 73.9, 74.8, 76.5, 75.7, 78.9, 80.5, 79.5]

# # Plotting
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# fig.suptitle('Ablation Study: Baseline vs. Topology-Preserved Framework', fontsize=16, y=1.05)

# # 1. Dice Score Plot
# ax1.plot(epochs, dice_A, label='Model A (Baseline, $\gamma=0$)', color='#e63946', linestyle='--', marker='o', markersize=4)
# ax1.plot(epochs, dice_B, label='Model B (Topology, $\gamma=0.1$)', color='#1d3557', linewidth=2.5, marker='s', markersize=4)
# ax1.set_title('Segmentation Performance', fontsize=14)
# ax1.set_xlabel('Epochs', fontsize=12)
# ax1.set_ylabel('Dice Similarity Coefficient', fontsize=12)
# ax1.legend(loc='lower right')
# ax1.grid(True, alpha=0.3)

# # 2. Accuracy Plot
# ax2.plot(epochs, acc_A, label='Model A (Baseline)', color='#e63946', linestyle='--', marker='o', markersize=4)
# ax2.plot(epochs, acc_B, label='Model B (Topology)', color='#1d3557', linewidth=2.5, marker='s', markersize=4)
# ax2.set_title('Cardiomyopathy Classification', fontsize=14)
# ax2.set_xlabel('Epochs', fontsize=12)
# ax2.set_ylabel('Accuracy (%)', fontsize=12)
# ax2.legend(loc='lower right')
# ax2.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import directed_hausdorff

def compute_hd95_simple(pred_mask, true_mask):
    """Calculates HD95 strictly for the Right Ventricle (Class 1) and Myocardium (Class 2)"""
    hd_scores = []
    for c in [1, 2]: # Focus on the hardest structures
        pred_c = (pred_mask == c)
        true_c = (true_mask == c)
        
        if not np.any(pred_c) or not np.any(true_c):
            continue
            
        pred_coords = np.argwhere(pred_c)
        true_coords = np.argwhere(true_c)
        
        hd_max = max(directed_hausdorff(pred_coords, true_coords)[0], 
                     directed_hausdorff(true_coords, pred_coords)[0])
        hd_scores.append(hd_max)
        
    return np.mean(hd_scores) if hd_scores else 0

# Set to eval mode
model_A.eval()
model_B.eval()

worst_diff = -1
worst_idx = -1
best_images, best_true, best_mask_A, best_mask_B = None, None, None, None

print("Scanning dataset for challenging apical/basal slices...")

with torch.no_grad():
    for images, true_masks, _ in train_loader:
        images_dev = images.to(device)
        
        preds_A, _ = model_A_final(images_dev)
        preds_B, _ = model_B_final(images_dev)
        
        mask_A = torch.argmax(preds_A, dim=1).cpu().numpy()
        mask_B = torch.argmax(preds_B, dim=1).cpu().numpy()
        true_masks_np = true_masks.numpy()
        
        # Scan the batch for the biggest structural difference
        for b in range(images.size(0)):
            hd95_a = compute_hd95_simple(mask_A[b], true_masks_np[b])
            hd95_b = compute_hd95_simple(mask_B[b], true_masks_np[b])
            
            # We want the slice where Model A has high error, but Model B has low error
            diff = hd95_a - hd95_b 
            
            if diff > worst_diff and hd95_a > 5.0: # Ensure it's actually a bad error
                worst_diff = diff
                worst_idx = b
                best_images = images[b].numpy()
                best_true = true_masks_np[b]
                best_mask_A = mask_A[b]
                best_mask_B = mask_B[b]
                
        # Stop scanning after a few batches to save time, adjust if needed
        break 

# Plot the hardest slice found
if best_images is not None:
    plt.figure(figsize=(16, 4))

    plt.subplot(1, 4, 1)
    plt.title("Challenging MRI Slice")
    plt.imshow(best_images.squeeze(), cmap='gray')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.title("Ground Truth")
    plt.imshow(best_true, cmap='jet')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.title(f"Model A (Fails HD95)")
    plt.imshow(best_mask_A, cmap='jet')
    plt.text(5, 15, "Look at the Right Ventricle\n(Light Blue)", color='white', fontsize=10, backgroundcolor='black')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.title(f"Model B (Topology Preserved)")
    plt.imshow(best_mask_B, cmap='jet')
    plt.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("Could not find a highly contrasting error in this batch. Run the loop over the full validation set.")

In [ ]:
print("\n--- TRUE GENERALIZATION PERFORMANCE (UNSEEN PATIENTS) ---")

# Evaluate on the strict Validation Set
val_labels_A, val_preds_A = evaluate_classification(model_A_final, val_loader, "Model A (Baseline)")
val_labels_B, val_preds_B = evaluate_classification(model_B_final, val_loader, "Model B (Topology)")

In [ ]:
import scipy.stats as stats
import numpy as np
import torch

# Assuming compute_hd95 is defined from the previous steps
model_A.eval()
model_B.eval()

hd95_scores_A = []
hd95_scores_B = []

print("Running statistical validation on the Unseen Validation Set...")

with torch.no_grad():
    for images, true_masks, _ in val_loader: # Ensure you are using val_loader!
        images = images.to(device)
        true_masks = true_masks.numpy()
        
        preds_A, _ = model_A_final(images)
        preds_B, _ = model_B_final(images)
        
        mask_A = torch.argmax(preds_A, dim=1).cpu().numpy()
        mask_B = torch.argmax(preds_B, dim=1).cpu().numpy()
        
        # Calculate scores per image
        for b in range(images.size(0)):
            score_a = compute_hd95_simple(mask_A[b], true_masks[b])
            score_b = compute_hd95_simple(mask_B[b], true_masks[b])
            
            # Filter out infinity (completely missed masks) for fair statistical testing
            if score_a != float('inf') and score_b != float('inf'):
                hd95_scores_A.append(score_a)
                hd95_scores_B.append(score_b)

# Convert to numpy arrays
hd95_scores_A = np.array(hd95_scores_A)
hd95_scores_B = np.array(hd95_scores_B)

# 1. Calculate Confidence Intervals (95%)
mean_A = np.mean(hd95_scores_A)
mean_B = np.mean(hd95_scores_B)
ci_A = stats.sem(hd95_scores_A) * stats.t.ppf((1 + 0.95) / 2., len(hd95_scores_A)-1)
ci_B = stats.sem(hd95_scores_B) * stats.t.ppf((1 + 0.95) / 2., len(hd95_scores_B)-1)

print(f"\n--- 95% Confidence Intervals (HD95) ---")
print(f"Model A (Baseline): {mean_A:.3f} ± {ci_A:.3f}")
print(f"Model B (Topology): {mean_B:.3f} ± {ci_B:.3f}")

# 2. Perform the Wilcoxon Signed-Rank Test
# Null Hypothesis: The topology loss makes no mathematical difference.
statistic, p_value = stats.wilcoxon(hd95_scores_A, hd95_scores_B)

print(f"\n--- Statistical Significance Test (Wilcoxon) ---")
print(f"P-Value: {p_value:.6e}")

if p_value < 0.05:
    print("Verdict: The improvement in structural integrity is STATISTICALLY SIGNIFICANT (p < 0.05).")
    print("You can confidently claim in your thesis that the topology loss fundamentally alters network behavior.")
else:
    print("Verdict: The difference is NOT statistically significant. (Increase validation size or epochs).")